# Prenche colunas do csv

In [20]:
import pandas as pd
from pathlib import Path
import bibtexparser

# Caminhos dos arquivos
csv_path = Path("accepted_articles.csv")
bib_path = Path("accepted_articles.bib")

# Carregar o CSV
df = pd.read_csv(csv_path)

# 1. Preencher células vazias nas colunas "size" e "model" com "n/a"
df["size"] = df["size"].fillna("n/a")
df["model"] = df["model"].fillna("n/a")

# 2. Associar DOIs com a chave do BibTeX
# Criar a coluna 'ref' se ainda não existir
if "ref" not in df.columns:
    df["ref"] = ""

# Carregar o arquivo .bib
with open(bib_path, "r", encoding="utf-8") as bib_file:
    bib_database = bibtexparser.load(bib_file)

# Criar dicionário DOI -> BibTeX key
doi_to_bibkey = {}
for entry in bib_database.entries:
    doi = entry.get("doi", "").strip().lower()
    if doi:
        doi_to_bibkey[doi] = entry["ID"]

# Preencher a coluna 'ref' usando os DOIs
for idx, row in df.iterrows():
    # if (not row["ref"] or row["ref"] == "") and pd.notna(row["doi"]):
    doi_key = str(row["doi"]).strip().lower()
    matched_key = doi_to_bibkey.get(doi_key)
    if matched_key:
        df.at[idx, "ref"] = matched_key

# Salvar o CSV atualizado
df.to_csv(csv_path, index=False)

df.head()

C:\Users\gugli\AppData\Local\Temp\ipykernel_10856\2036218337.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Shen_2002_Sharing' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "ref"] = matched_key


,ref,title,year,source,selection_criteria,status,display,input,interaction,collaboration,model,size,doi
0,Shen_2002_Sharing,Sharing and building digital group histories,2002,ACM Digital Library,Study describing digital tabletop application ...,Accepted,Front-Projected,Pen,Tangibility,Colaboração presencial,n/a,n/a,10.1145/587078.587124
1,Vernier_2002_Visualization,Visualization techniques for circular tabletop...,2002,Scopus,Study describing digital tabletop techniques,Accepted,Front-Projected,"Pen, Sound-Based",Tangibility,Colaboração presencial,n/a,n/a,10.1145/1556262.1556305
2,Shen_2003_Ubitable,UbiTable: Impromptu face-to-face collaboration...,2003,Scopus,Study describing digital tabletop techniques,Accepted,Front-Projected,Electrical-Based,"Multi-touch, Touch",Colaboração presencial,DiamondTouch,42”,10.1007/978-3-540-39653-6_22
3,Shen_2003_Personal,Personal digital historian: story sharing arou...,2003,ACM Digital Library,Study describing digital tabletop application ...,Accepted,Front-Projected,Electrical-Based,"Multi-touch, Touch",Colaboração presencial,DiamondTouch,42”,10.1145/637848.637856
4,Moghaddam_2004_Visualization,Visualization and User-Modeling for Browsing P...,2004,Springer Link,Study describing digital tabletop techniques,Accepted,Front-Projected,Pen,Tangibility,Colaboração presencial,n/a,n/a,10.1023/b:visi.0000004834.62090.74


# Adiciona organization no bib

In [22]:
from pathlib import Path
import bibtexparser

# Caminho para o arquivo .bib
bib_path = Path("accepted_articles.bib")

# Carregar o arquivo .bib
with open(bib_path, "r", encoding="utf-8") as f:
    bib_database = bibtexparser.load(f)

# Atualizar entradas faltando 'organization'
for entry in bib_database.entries:
    entry_type = entry.get("ENTRYTYPE", "").lower()
    if entry_type in {"inproceedings", "article"} and "organization" not in entry:
        entry["organization"] = "{}"  # Valor vazio

# Salvar o .bib atualizado
with open(bib_path, "w", encoding="utf-8") as f:
    bibtexparser.dump(bib_database, f)

print("✔ Campos 'organization' adicionados onde estavam ausentes.")


✔ Campos 'organization' adicionados onde estavam ausentes.


# Cria a tabela latex

In [ ]:
import pandas as pd
from pathlib import Path

# Caminho para o CSV
csv_path = Path("accepted_articles.csv")

# Carregar os dados
df = pd.read_csv(csv_path)

# Selecionar e renomear as colunas relevantes
df_table = df[["ref", "display", "input", "interaction", "collaboration", "model", "size"]].copy()
df_table.columns = ["Referência", "Display", "Input", "Interação", "Colaboração", "Modelo", "Tamanho"]

# Função para escapar caracteres especiais no LaTeX
def latex_escape(text):
    if pd.isna(text):
        return "n/a"
    return str(text).replace('&', r'\&').replace('%', r'\%').replace('_', r'\_').replace('$', r'\$')

# Escapar todas as colunas, exceto "Referência"
for col in df_table.columns:
    if col != "Referência":
        df_table[col] = df_table[col].apply(latex_escape)

# Gerar o conteúdo LaTeX
latex_lines = []
latex_lines.append("\\begin{supertabular}{")
latex_lines.append("    p{0.08\\textwidth} % Referencia")
latex_lines.append("    p{0.15\\textwidth} % Display")
latex_lines.append("    p{0.20\\textwidth} % Input")
latex_lines.append("    p{0.15\\textwidth} % Interação")
latex_lines.append("    p{0.10\\textwidth} % Coluna 5 (e.g., Colaboração Suportada)")
latex_lines.append("    p{0.10\\textwidth} % Coluna 6 (e.g., Modelo Utilizado)")
latex_lines.append("    p{0.08\\textwidth} % Coluna 7 (e.g., Tamanho da Superfície Interativa)")
latex_lines.append("}")

latex_lines.append("\\toprule") # Top rule

latex_lines.append("\\textbf{Referência} & \\textbf{Display} & \\textbf{Input} & \\textbf{Interação} & \\textbf{Colaboração} & \\textbf{Modelo} & \\textbf{Tamanho} \\\\")
latex_lines.append("\\toprule") # Top rule

# Adicionar linhas da tabela
for _, row in df_table.iterrows():
    row["Referência"] = f"\\cite{{{row['Referência'].strip()}}}"
    line = " & ".join(row.values) + " \\\\"
    latex_lines.append(line)
    latex_lines.append('\\hline')


latex_lines.append("\\bottomrule % Final bottom rule")
latex_lines.append("\\end{supertabular}")

# Salvar como .tex
output_path = Path("tabela_artigos.tex")
with open(output_path, "w", encoding="utf-8") as f:
    f.write("\n".join(latex_lines))

# Mostrar as primeiras linhas da tabela LaTeX gerada
latex_preview = "\n".join(latex_lines[:15])
latex_preview


'\\begin{supertabular}{\n    p{0.08\\textwidth} % Referencia\n    p{0.15\\textwidth} % Display\n    p{0.20\\textwidth} % Input\n    p{0.15\\textwidth} % Interação\n    p{0.10\\textwidth} % Coluna 5 (e.g., Colaboração Suportada)\n    p{0.10\\textwidth} % Coluna 6 (e.g., Modelo Utilizado)\n    p{0.08\\textwidth} % Coluna 7 (e.g., Tamanho da Superfície Interativa)\n}\n\\toprule\n\\textbf{Referência} & \\textbf{Display} & \\textbf{Input} & \\textbf{Interação} & \\textbf{Colaboração} & \\textbf{Modelo} & \\textbf{Tamanho} \\\\\n\\toprule\n\\cite{Shen_2002_Sharing} & Front-Projected & Pen & Tangibility & Colaboração presencial & n/a & n/a \\\\\n\\hline\n\\cite{Vernier_2002_Visualization} & Front-Projected & Pen, Sound-Based & Tangibility & Colaboração presencial & n/a & n/a \\\\'